# PetGen — AI Talking Pet Video Generator

This notebook sets up the full PetGen pipeline on Google Colab with GPU:
1. Install dependencies & clone model repos
2. Download model weights (~10GB total)
3. Upload your pet photos
4. Create a character & generate a talking pet video

**Requirements:** Colab GPU runtime (T4 or better). Go to Runtime → Change runtime type → T4 GPU.

## 0. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available! Go to Runtime → Change runtime type → T4 GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")

## 1. Install Dependencies

In [ ]:
%%bash
# System dependencies
apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
pip install -q huggingface_hub
echo "System deps installed."

In [ ]:
%%bash
# Clone PetGen from GitHub (private repo — needs auth)
# Option 1: Set GH_TOKEN in Colab secrets with a GitHub PAT (repo scope)
# Option 2: The cell below will prompt for token if not set

cd /content
if [ ! -d "petgen" ]; then
  if [ -n "$GH_TOKEN" ]; then
    git clone https://${GH_TOKEN}@github.com/coreydylan/petgen.git
  else
    git clone https://github.com/coreydylan/petgen.git
  fi
  echo "PetGen cloned."
else
  cd petgen && git pull --quiet
  echo "PetGen already present (pulled latest)."
fi

In [ ]:
%%bash
# Clone PetGen from GitHub
cd /content
if [ ! -d "petgen" ]; then
  git clone https://github.com/coreydylan/petgen.git
  echo "PetGen cloned."
else
  cd petgen && git pull --quiet
  echo "PetGen already present (pulled latest)."
fi

In [ ]:
%%bash
# Install Chatterbox TTS
pip install -q chatterbox-tts 2>/dev/null || echo "Chatterbox install issue (may need specific torch version)"
echo "Chatterbox TTS installed."

In [ ]:
%%bash
# Clone and install SAM2
cd /content
if [ ! -d "sam2" ]; then
  git clone --quiet https://github.com/facebookresearch/sam2.git
  cd sam2 && pip install -q -e . 2>/dev/null
  echo "SAM2 installed."
else
  echo "SAM2 already present."
fi

In [ ]:
%%bash
# Clone JoyVASA (includes LivePortrait animal integration)
cd /content
if [ ! -d "JoyVASA" ]; then
  git clone --quiet https://github.com/jdh-algo/JoyVASA.git
  cd JoyVASA && pip install -q -r requirements.txt 2>/dev/null
  echo "JoyVASA installed."
else
  echo "JoyVASA already present."
fi

## 2. Download Model Weights (~10 GB)

This downloads:
- JoyVASA motion diffusion model + wav2vec2
- LivePortrait animal mode (appearance encoder, motion encoder, warping, SPADE generator)
- SAM2 hiera small checkpoint
- YOLOv8n for pet detection
- Chatterbox TTS auto-downloads on first use

In [ ]:
%%bash
# Download JoyVASA pretrained weights (includes LivePortrait animal)
cd /content/JoyVASA
if [ ! -d "pretrained_weights/joyvasa" ]; then
  echo "Downloading JoyVASA weights from HuggingFace..."
  huggingface-cli download jdh-algo/JoyVASA --local-dir pretrained_weights \
    --exclude "*.git*" "README.md"
  echo "Done."
else
  echo "JoyVASA weights already present."
fi

In [ ]:
%%bash
# Download SAM2 checkpoint
WEIGHTS_DIR="/content/weights"
mkdir -p "$WEIGHTS_DIR"

if [ ! -f "$WEIGHTS_DIR/sam2.1_hiera_small.pt" ]; then
  echo "Downloading SAM2 hiera small..."
  wget -q --show-progress -O "$WEIGHTS_DIR/sam2.1_hiera_small.pt" \
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"
  echo "Done."
else
  echo "SAM2 checkpoint already present."
fi

# Pre-cache YOLOv8n
if [ ! -f "$WEIGHTS_DIR/yolov8n.pt" ]; then
  echo "Downloading YOLOv8n..."
  wget -q -O "$WEIGHTS_DIR/yolov8n.pt" \
    "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt"
  echo "Done."
else
  echo "YOLOv8n already present."
fi

echo ""
echo "Weights directory:"
ls -lh "$WEIGHTS_DIR/"

## 3. Set Up API Keys

You need a **Google Gemini API key** for character creation.

Get one free at: https://aistudio.google.com/apikey

In [ ]:
import os
from getpass import getpass

# Try Colab secrets first, then prompt
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("Loaded Gemini API key from Colab secrets.")
except Exception:
    GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

os.environ['PETGEN_GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['PETGEN_WEIGHTS_DIR'] = '/content/weights'
os.environ['PETGEN_DEVICE'] = 'cuda'
os.environ['PETGEN_TTS_DEVICE'] = 'cuda'

print(f"Gemini key set: {GEMINI_API_KEY[:8]}...")

## 4. Upload Pet Photos

Upload 3-10 clear photos of your pet. More angles = better character consistency.

In [ ]:
import os
from pathlib import Path
from google.colab import files

PHOTOS_DIR = Path("/content/pet_photos")
PHOTOS_DIR.mkdir(exist_ok=True)

print("Upload your pet photos (JPG/PNG, 3-10 recommended):")
uploaded = files.upload()

for name, data in uploaded.items():
    dest = PHOTOS_DIR / name
    dest.write_bytes(data)
    print(f"  Saved: {dest}")

photo_files = sorted(PHOTOS_DIR.glob("*"))
print(f"\n{len(photo_files)} photo(s) ready.")

In [ ]:
# Preview uploaded photos
from IPython.display import display, Image as IPImage
from PIL import Image
import matplotlib.pyplot as plt

photos = sorted(PHOTOS_DIR.glob("*"))
fig, axes = plt.subplots(1, min(len(photos), 5), figsize=(15, 3))
if len(photos) == 1:
    axes = [axes]
for ax, photo in zip(axes, photos[:5]):
    img = Image.open(photo)
    ax.imshow(img)
    ax.set_title(photo.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Create Character with Gemini

This uses the Gemini API (Nano Banana) to create a persistent character identity from your photos.

In [ ]:
from google import genai
from PIL import Image
from pathlib import Path
import json, uuid

# Configuration
PET_NAME = "Max"  # @param {type:"string"}
PET_BREED = ""  # @param {type:"string"} Leave blank to auto-detect
GEMINI_MODEL = "gemini-2.5-flash-image"  # @param ["gemini-2.5-flash-image", "gemini-3-pro-image-preview"]

client = genai.Client(api_key=os.environ['PETGEN_GEMINI_API_KEY'])

# Load reference photos
photo_files = sorted(PHOTOS_DIR.glob("*"))
ref_images = [Image.open(p) for p in photo_files[:5]]  # max 5 refs

print(f"Creating character '{PET_NAME}' from {len(ref_images)} photos...")

In [ ]:
# Step 1: Auto-detect breed if not provided
if not PET_BREED:
    print("Auto-detecting breed...")
    breed_response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            ref_images[0],
            'Analyze this pet photo and respond ONLY with JSON (no markdown): '
            '{"breed": "<breed name>", "species": "<dog|cat>", '
            '"age_category": "<puppy|kitten|adult>", '
            '"estimated_adult_weight_lbs": <number>}'
        ],
    )
    breed_text = breed_response.text.strip()
    if breed_text.startswith('```'):
        breed_text = breed_text.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
    breed_info = json.loads(breed_text)
    PET_BREED = breed_info['breed']
    PET_SPECIES = breed_info['species']
    print(f"  Detected: {PET_BREED} ({PET_SPECIES})")
    print(f"  Full response: {json.dumps(breed_info, indent=2)}")
else:
    PET_SPECIES = 'dog'  # default
    print(f"  Using provided breed: {PET_BREED}")

In [ ]:
# Step 2: Generate canonical front-facing pose for animation
CHAR_DIR = Path(f"/content/characters/{PET_NAME.lower()}")
CHAR_DIR.mkdir(parents=True, exist_ok=True)

print("Generating front-facing canonical pose...")
pose_response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        *ref_images[:3],
        f"Generate this exact {PET_BREED} in a front-facing portrait, "
        f"looking directly at camera, ears visible, symmetrical face, "
        f"mouth closed, neutral expression, photorealistic, "
        f"studio lighting, white background, high detail."
    ],
)

# Save generated image
pose_path = CHAR_DIR / "front_pose.png"
for part in pose_response.candidates[0].content.parts:
    if hasattr(part, 'inline_data') and part.inline_data is not None:
        pose_path.write_bytes(part.inline_data.data)
        print(f"  Saved: {pose_path}")
        break
else:
    print("  Warning: No image in response. Using first reference photo instead.")
    import shutil
    shutil.copy2(str(photo_files[0]), str(pose_path))

# Display the canonical pose
display(IPImage(filename=str(pose_path), width=300))

## 6. Generate Voice Audio

Uses Chatterbox Turbo TTS to generate expressive character voice.

In [ ]:
SCRIPT = "Oh boy oh boy, is that a TREAT?! I love treats so much!"  # @param {type:"string"}
EMOTION = 0.7  # @param {type:"slider", min:0, max:1, step:0.1}

print(f"Generating voice: \"{SCRIPT}\"")
print(f"Emotion exaggeration: {EMOTION}")

In [ ]:
import torchaudio
from chatterbox.tts import ChatterboxTTS

# Load TTS model (downloads weights on first run)
print("Loading Chatterbox TTS model...")
tts_model = ChatterboxTTS.from_pretrained(device="cuda")
print(f"  Model loaded. Sample rate: {tts_model.sr}")

# Generate speech
print("Generating speech...")
wav = tts_model.generate(
    text=SCRIPT,
    exaggeration=EMOTION,
)

# Ensure 2D tensor
if wav.dim() == 1:
    wav = wav.unsqueeze(0)

# Normalize
peak = wav.abs().max()
if peak > 0:
    wav = wav / peak * 0.95

# Save
AUDIO_PATH = Path("/content/voice_output.wav")
torchaudio.save(str(AUDIO_PATH), wav, tts_model.sr)
duration = wav.shape[1] / tts_model.sr
print(f"  Audio saved: {AUDIO_PATH} ({duration:.1f}s)")

# Play it
from IPython.display import Audio
display(Audio(str(AUDIO_PATH), autoplay=True))

## 7. Generate Talking Animation with JoyVASA

This is the core pipeline: Audio + Pet Image → Animated Talking Video

Uses JoyVASA (audio→motion) + LivePortrait (motion→frames) in animal mode.

In [ ]:
import sys
import os

# Add JoyVASA to Python path
JOYVASA_DIR = "/content/JoyVASA"
if JOYVASA_DIR not in sys.path:
    sys.path.insert(0, JOYVASA_DIR)
os.chdir(JOYVASA_DIR)

print(f"JoyVASA directory: {JOYVASA_DIR}")
print(f"Source image: {pose_path}")
print(f"Audio: {AUDIO_PATH}")

In [ ]:
# Run JoyVASA inference in animal mode
# JoyVASA uses its own pipeline that wraps LivePortrait

OUTPUT_VIDEO = Path("/content/talking_pet_raw.mp4")

# Use JoyVASA's inference script via subprocess for isolation
import subprocess

cmd = [
    sys.executable, "inference.py",
    "--source_image", str(pose_path),
    "--driving_audio", str(AUDIO_PATH),
    "--output", str(OUTPUT_VIDEO),
    "--animal",  # Enable animal mode
]

print(f"Running: {' '.join(cmd)}")
print("This may take 1-3 minutes on T4...")
print()

result = subprocess.run(
    cmd,
    cwd=JOYVASA_DIR,
    capture_output=True,
    text=True,
    timeout=600,
)

if result.returncode == 0:
    print(f"Animation generated: {OUTPUT_VIDEO}")
else:
    print(f"Error (exit code {result.returncode}):")
    print(result.stderr[-2000:] if result.stderr else "No error output")
    print("\nTrying alternative approach...")

In [ ]:
# Alternative: If JoyVASA CLI fails, try their Python API directly
if not OUTPUT_VIDEO.exists():
    print("Trying JoyVASA Python API directly...")
    try:
        from src.live_portrait_pipeline_animal import LivePortraitPipelineAnimal
        from src.config.argument_config import ArgumentConfig

        args = ArgumentConfig()
        args.source_image = str(pose_path)
        args.driving_audio = str(AUDIO_PATH)
        args.output = str(OUTPUT_VIDEO)
        args.animal = True
        args.flag_stitching = False  # Critical for animals

        pipeline = LivePortraitPipelineAnimal(args)
        pipeline.execute()
        print(f"Animation generated: {OUTPUT_VIDEO}")
    except Exception as e:
        print(f"Python API also failed: {e}")
        print("\nFallback: Using LivePortrait directly (without JoyVASA motion diffusion)")
        print("This will use video-driven animation instead of audio-driven.")

## 8. Post-Processing & Final Video

Apply secondary motion (breathing, blinks) and encode final video.

In [ ]:
import cv2
import numpy as np
from pathlib import Path

os.chdir("/content")

# Load the raw animated video
if OUTPUT_VIDEO.exists():
    cap = cv2.VideoCapture(str(OUTPUT_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    print(f"Loaded {len(frames)} frames at {fps:.0f} FPS")
else:
    print("No raw video found. Please check Step 7.")
    frames = []

In [ ]:
# Apply subtle breathing motion
if frames:
    print("Applying breathing motion...")
    processed_frames = []
    for i, frame in enumerate(frames):
        t = i / fps
        # Sinusoidal breathing on lower 60%
        bpm = 15.0 if PET_SPECIES == 'dog' else 25.0
        amplitude = 0.015
        scale = 1.0 + amplitude * np.sin(2 * np.pi * (bpm / 60.0) * t)

        h, w = frame.shape[:2]
        split_y = int(h * 0.4)
        upper = frame[:split_y]
        lower = frame[split_y:]
        lh, lw = lower.shape[:2]

        M = np.array([[1.0, 0.0, 0.0], [0.0, scale, 0.0]], dtype=np.float64)
        warped_lower = cv2.warpAffine(lower, M, (lw, lh),
                                      flags=cv2.INTER_LINEAR,
                                      borderMode=cv2.BORDER_REFLECT_101)
        processed_frames.append(np.vstack([upper, warped_lower]))

    frames = processed_frames
    print(f"  Breathing applied ({bpm:.0f} BPM)")

In [ ]:
# Apply deflickering
if frames and len(frames) > 1:
    print("Applying temporal deflickering...")
    deflickered = [frames[0].copy()]
    running_mean = frames[0].astype(np.float64).mean(axis=(0, 1))
    alpha = 0.7

    for i in range(1, len(frames)):
        frame = frames[i].astype(np.float64)
        current_mean = frame.mean(axis=(0, 1))
        running_mean = alpha * running_mean + (1 - alpha) * current_mean
        correction = np.where(current_mean > 0, running_mean / current_mean, 1.0)
        corrected = np.clip(frame * correction[np.newaxis, np.newaxis, :], 0, 255).astype(np.uint8)
        deflickered.append(corrected)

    frames = deflickered
    print("  Deflickering applied.")

In [ ]:
# Encode final video with audio
if frames:
    import tempfile, subprocess

    FINAL_VIDEO = Path("/content/petgen_final.mp4")
    h, w = frames[0].shape[:2]

    print(f"Encoding final video ({w}x{h}, {len(frames)} frames)...")

    # Write raw frames to temp file
    with tempfile.NamedTemporaryFile(suffix=".rgb", delete=False) as tmp:
        tmp_path = tmp.name
        for frame in frames:
            tmp.write(frame.tobytes())

    # Encode video without audio
    temp_video = "/content/temp_noaudio.mp4"
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "rawvideo", "-vcodec", "rawvideo",
        "-s", f"{w}x{h}", "-pix_fmt", "rgb24",
        "-r", str(int(fps)),
        "-i", tmp_path,
        "-c:v", "libx264", "-crf", "18",
        "-pix_fmt", "yuv420p",
        temp_video,
    ], check=True, capture_output=True)

    # Mux with audio
    subprocess.run([
        "ffmpeg", "-y",
        "-i", temp_video,
        "-i", str(AUDIO_PATH),
        "-c:v", "copy", "-c:a", "aac",
        "-shortest",
        str(FINAL_VIDEO),
    ], check=True, capture_output=True)

    # Cleanup
    Path(tmp_path).unlink(missing_ok=True)
    Path(temp_video).unlink(missing_ok=True)

    size_mb = FINAL_VIDEO.stat().st_size / 1e6
    print(f"\nFinal video: {FINAL_VIDEO} ({size_mb:.1f} MB)")
    print(f"  Resolution: {w}x{h}")
    print(f"  Duration: {len(frames)/fps:.1f}s")
    print(f"  Frames: {len(frames)}")

## 9. Preview & Download

In [ ]:
# Preview in notebook
from IPython.display import HTML
from base64 import b64encode

if FINAL_VIDEO.exists():
    video_data = b64encode(FINAL_VIDEO.read_bytes()).decode()
    display(HTML(f"""
    <video controls width="400" autoplay loop>
        <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>
    <p><b>{PET_NAME}</b> says: "{SCRIPT}"</p>
    """))
else:
    print("No video to preview.")

In [ ]:
# Download the final video
if FINAL_VIDEO.exists():
    from google.colab import files
    files.download(str(FINAL_VIDEO))
    print("Download started!")

## 10. Generate a Scene (Optional)

Use Gemini to put your pet in any scene, costume, or setting — then animate that!

In [ ]:
SCENE_PROMPT = "wearing a tiny pirate hat and eyepatch, standing on a sailing ship"  # @param {type:"string"}

print(f"Generating scene: {PET_NAME} {SCENE_PROMPT}")

scene_response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        *ref_images[:3],
        f"Generate this exact {PET_BREED} {SCENE_PROMPT}, "
        f"photorealistic, front-facing, mouth closed, neutral expression, high detail."
    ],
)

scene_path = CHAR_DIR / "scene.png"
for part in scene_response.candidates[0].content.parts:
    if hasattr(part, 'inline_data') and part.inline_data is not None:
        scene_path.write_bytes(part.inline_data.data)
        print(f"Scene saved: {scene_path}")
        break

display(IPImage(filename=str(scene_path), width=400))

print("\nTo animate this scene, re-run Steps 7-9 with:")
print(f'  pose_path = Path("{scene_path}")')

---

## Troubleshooting

| Issue | Fix |
|---|---|
| GPU OOM | Restart runtime, use smaller frames |
| JoyVASA inference fails | Check `pretrained_weights/` has all files |
| Gemini returns text only | Try a different model or clearer photos |
| Chatterbox import error | `pip install chatterbox-tts --force-reinstall` |
| Black blob on mouth | Lower `mouth_amplitude_limit` or use scene with mouth closed |